In [43]:

# Initialisation
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Chargement des données (Iceberg)
df_hourly = spark.table("iceberg.gold.hourly_patterns")
df_weather = spark.table("iceberg.gold.weather_impact")

# 2. Jointure pour enrichir les données
# On joint sur la date pour récupérer la météo
dataset = df_hourly.join(df_weather, "pickup_date", "inner")

# 3. Préparation des Features
input_cols = ["pickup_hour", "is_rainy"] # Sélection dynamique
assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
final_data = assembler.transform(dataset)

# 4. Split Train/Test
(training_data, test_data) = final_data.randomSplit([0.8, 0.2])

# 5. Configuration du Modèle (Random Forest)
rf = RandomForestRegressor(featuresCol="features", labelCol="trip_count", maxDepth=5, numTrees=20)

# 6. Entraînement
model = rf.fit(training_data)

# 7. Prédictions & Évaluation
predictions = model.transform(test_data)
evaluator = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")
display(predictions.select("pickup_date", "pickup_hour", "trip_count", "prediction"))
                                     

26/01/20 16:34:22 ERROR Executor: Exception in task 4.0 in stage 36.0 (TID 89)5]
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`VectorAssembler$$Lambda$5006/0x00007f172549d000`: (struct<pickup_hour_double_VectorAssembler_53a63a0eff02:double,is_rainy_double_VectorAssembler_53a63a0eff02:double>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:198)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.sort_addToSorter_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.j

Py4JJavaError: An error occurred while calling o295.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 4 in stage 36.0 failed 1 times, most recent failure: Lost task 4.0 in stage 36.0 (TID 89) (d9a1784cf332 executor driver): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`VectorAssembler$$Lambda$5006/0x00007f172549d000`: (struct<pickup_hour_double_VectorAssembler_53a63a0eff02:double,is_rainy_double_VectorAssembler_53a63a0eff02:double>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:198)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.sort_addToSorter_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.e